In [29]:
#download pdfs
import csv
import re
import urllib.request
from pathlib import Path

CSV_PATH = "books.csv"
OUTDIR = Path("books")

def slugify(title: str) -> str:
    slug = title.strip().replace(" ", "_")
    return re.sub(r"[^A-Za-z0-9_-]", "", slug)

OUTDIR.mkdir(exist_ok=True)

with open(CSV_PATH, newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        title = row["title"]
        pdf_url = row["pdf_url"]
        dest = OUTDIR / f"{slugify(title)}.pdf"

        if dest.exists():
            print(f"Skipping (already exists): {dest}")
            continue

        print(f"Downloading: {title} -> {dest}")
        try:
            urllib.request.urlretrieve(pdf_url, dest)
        except Exception as e:
            print(f"  FAILED: {pdf_url} ({e})")

print("\nDone. Files in", OUTDIR)
for p in sorted(OUTDIR.iterdir()):
    print(" ", p.name, p.stat().st_size, "bytes")

Skipping (already exists): books/Think_Python_2e.pdf
Skipping (already exists): books/Think_DSP.pdf
Skipping (already exists): books/Think_Complexity_2e.pdf
Skipping (already exists): books/Think_Java_2e.pdf
Skipping (already exists): books/Physical_Modeling_in_MATLAB.pdf
Skipping (already exists): books/Think_OS.pdf
Skipping (already exists): books/Think_C.pdf

Done. Files in books
  Physical_Modeling_in_MATLAB.pdf 1313152 bytes
  Think_C.pdf 978841 bytes
  Think_Complexity_2e.pdf 6294093 bytes
  Think_DSP.pdf 4275659 bytes
  Think_Java_2e.pdf 1957646 bytes
  Think_OS.pdf 368105 bytes
  Think_Python_2e.pdf 921415 bytes


In [30]:
#Q1 convert pdf to markdown and move to books_text
from markitdown import MarkItDown
from pathlib import Path
import os
out_dir= Path("books_text")
input_dir=Path("books")

md=MarkItDown()

files= input_dir.iterdir()
dest=""
for f in files:
    if f.is_file() and f.suffix.lower()==".pdf":
        dest= out_dir /(f.stem + ".md")
        if dest.exists():
            print(f"Skipping (already exists): {dest}")
            continue
        print(f"Converting: {f.name} -> {dest}")
        try:
            result=md.convert(str(f))
            dest.write_text(result.text_content,encoding="utf-8")
            count = 0
            with open(dest, encoding="utf-8") as file:
                for line in file:
                    count += 1
            num_lines = count
            print(f"  {dest.name}: {num_lines} lines")
        except Exception as e:
            print(f"Failed : {f.name} ({e})")

print("\nDone. Files in", out_dir)
""" for p in sorted(out_dir.iterdir()):
    print(" ", p.name, p.stat().st_size, "bytes") """

Skipping (already exists): books_text/Think_OS.md
Skipping (already exists): books_text/Think_DSP.md
Skipping (already exists): books_text/Think_Java_2e.md
Skipping (already exists): books_text/Think_C.md
Skipping (already exists): books_text/Think_Complexity_2e.md
Skipping (already exists): books_text/Think_Python_2e.md
Skipping (already exists): books_text/Physical_Modeling_in_MATLAB.md

Done. Files in books_text


' for p in sorted(out_dir.iterdir()):\n    print(" ", p.name, p.stat().st_size, "bytes") '

In [31]:
#Q2 chunking for RAG
from pathlib import Path
from gitsource.chunking import chunk_documents
books_text_dir=Path("books_text")
documents = []
for f in sorted(books_text_dir.iterdir()):
    if f.is_file():
        with f.open(encoding="utf-8") as file:
            lines = [line.rstrip("\n") for line in file if line.strip()]
        documents.append({"source": f.name, "content": lines})
""" for doc in documents[:5]:
    for line in doc["content"][:3]:
        print(line) """


#chunking
chunks = chunk_documents(documents, size=100, step=50)
""" print(len(chunks), "chunks created")
for chunk in chunks[:100]:
    print(chunk.keys())
    print("source:", chunk["source"])
    print("start:", chunk["start"])
    print("content :", chunk["content"])
    print("---") """


think_python_chunks = [c for c in chunks if "Think_Python" in c["source"]]
print(len(think_python_chunks))



189


In [32]:
#Q3 index 
from minsearch import Index
from pprint import pprint

def prepare_documents(chunks):
    prepared=[]
    for chunk in chunks:
        chunk_copy=chunk.copy()
        chunk_copy["content"]="\n".join(chunk_copy["content"])
        prepared.append(chunk_copy)
    return prepared
    
documents = prepare_documents(chunks)

index = Index(text_fields=["content"])
index.fit(documents)

pprint(len(documents))



988


In [33]:
#Q4 searching and RAG
results = index.search("python function definition", num_results=5)
pprint (results)

[{'content': '| ------ | ------------ | --- | --- |\n'
             '| total  | = total +    | a;  |     |\n'
             'end\n'
             'ans = total\n'
             'Ifyouwereusinganyofthosevariablenamesbeforecallingthisscript, '
             'youmightbesurprised\n'
             'to find, after running the script, that their values had '
             'changed. If you have two scripts that\n'
             '48 Functions\n'
             'use the same variable names, you might find that they work '
             'separately and then break when\n'
             'you try to combine them. This kind of interaction is called a '
             'collision.\n'
             'name\n'
             'As the number of scripts you write increases, and they get '
             'longer and more complex, name\n'
             'collisions become more of a problem. Avoiding this problem is '
             'one of several motivations for\n'
             'functions.\n'
             '| 5.2 | Defining | Functio

In [34]:
#Q5 Full RAG
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()
import json

instructions = """
You're a course assistant, your task is to answer the QUESTION from the
course students using the provided CONTEXT
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    prompt = prompt_template.format(
        question=question,
        context=context
    ).strip()
    return prompt

def search(question):
    return index.search(question, num_results=5)


def llm(user_prompt, instructions, model='gpt-4o-mini'):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=messages
    )
    
    #return response.output_text
    return {
        "answer":response.output_text,
        "input_tokens": response.usage.input_tokens,
        "output_tokens" : response.usage.output_tokens
    }

def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer


query="python function definition"
answer = rag(query)
pprint (answer)

{'answer': 'In Python, a function is defined using the `def` keyword followed '
           'by the function name and parentheses containing any parameters. '
           'The basic structure looks like this:\n'
           '\n'
           '```python\n'
           'def function_name(parameters):\n'
           '    # Code block\n'
           '    return output\n'
           '```\n'
           '\n'
           '### Example:\n'
           '\n'
           'Here’s a simple function that takes two numbers as parameters and '
           'returns their sum:\n'
           '\n'
           '```python\n'
           'def add_numbers(a, b):\n'
           '    return a + b\n'
           '```\n'
           '\n'
           '### Key Points:\n'
           '- **Function Name**: It should be descriptive enough to convey '
           'what the function does (e.g., `add_numbers`).\n'
           '- **Parameters**: The inputs to the function; they can be multiple '
           'or none.\n'
           '- **Code Bloc

In [35]:
#Question 6. Structured vs Unstructured Output
from pydantic import BaseModel, Field
from typing import Literal

class RAGResponse(BaseModel):
    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions")


""" Modify the llm and rag functions from Question 5 to use structured outputs.

do RAG for "python function definition"
look at the number of input tokens
compare the number with the results from Q5
How many MORE input tokens does the structured output version use compared to the unstructured version? """

def llm_structured(user_prompt, instructions, model='gpt-4o-mini', output_type=RAGResponse):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.parse(
        model=model,
        input=messages,
        text_format=output_type
    )
    
    #return response.output_text
    return {
        "answer":response.output_text,
        "input_tokens": response.usage.input_tokens,
        "output_tokens" : response.usage.output_tokens
    }

def rag_structured(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm_structured(prompt, instructions)
    return answer

query="python function definition"
answer = rag_structured(query)
pprint (answer)


{'answer': '{"answer":"In Python, a function definition is a block of code '
           'that performs a specific task. You define a function using the '
           '`def` keyword followed by the function name and parentheses. '
           'Inside the parentheses, you can specify parameters (inputs) to '
           'your function. Below is the basic syntax for defining a function '
           'in Python:\\n\\n```python\\ndef function_name(parameters):\\n    # '
           'Code block that performs the task\\n    return output  # Optional '
           'return statement\\n```\\n\\n### Example:\\nHere is a simple '
           'example of a function that calculates the square of a '
           'number:\\n\\n```python\\ndef square(x):\\n    return x ** '
           '2\\n```\\n\\nYou can call this function with a number like '
           'so:\\n\\n```python\\nresult = square(5)\\nprint(result)  # Output '
           'will be 25\\n```\\n\\n### Key Points:\\n- **Function Name**: '
           '

In [36]:
#Homework answers
""" Question 1
9,703

Question 2
189

Question 3
988

Question 4
Physical Modeling in MATLAB

Question 5
7944

Question 6
186

Learning in Public
Not answered """


' Question 1\n9,703\n\nQuestion 2\n189\n\nQuestion 3\n988\n\nQuestion 4\nPhysical Modeling in MATLAB\n\nQuestion 5\n7944\n\nQuestion 6\n186\n\nLearning in Public\nNot answered '